In [ ]:
import os 
import requests # HTTP istemcisi
from PIL import Image # Pillow kütüphanesi
from openai import OpenAI  

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  
)

try:
    user_prompt = 'batman'
    
    print("[İŞLEM 1] Llama 3.2 ile görsel promptu optimize ediliyor...")
    
    response = client.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": "You are an expert prompt engineer for Stable Diffusion. Expand the given prompt with artistic details, style, lighting, and colors. Keep your response concise and separated by commas. Do not write any introduction or explanation."},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    optimized_prompt = response.choices[0].message.content
    print(f"[LLAMA 3.2 PROMPT]: {optimized_prompt}")

    print("[İŞLEM 2] Görsel API üzerinden üretiliyor...")
    
    image_url = f"https://image.pollinations.ai/p/{requests.utils.quote(optimized_prompt)}?width=512&height=512&seed=42"

    # requests.utils.quote -> URL encoding işlemi yapar

    image_dir = os.path.join(os.curdir, 'images')
    if not os.path.isdir(image_dir):
        os.mkdir(image_dir)

    image_path = os.path.join(image_dir, 'generated-image.png')

    generated_image = requests.get(image_url).content  
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)

    print(f"[BAŞARI] Görsel kaydedildi: {image_path}")

    # Notebook'ta görseli doğrudan hücre altında görmek için show() yerine bu satırı kullanabiliriz:
    image = Image.open(image_path)
    image.show()

except Exception as err:
    print(f"Bir hata oluştu: {err}")